# DS 310 - Homework 2: Build & Query the Insighta Database

**Name:** Berk Gozek

---

## Overview
In this homework you will **build** the Insighta database from raw CSV files and then **query** it.

1. **Part 1 - Create the schema.** Write `CREATE TABLE` statements for all five tables.
2. **Part 2 - Load the data.** Run the **provided** Python loader cell that fills your tables from the CSVs in `data/` (nothing to write, not graded).
3. **Part 3 - Query the database.** Answer 11 query questions against the database you built.
4. **Part 4 - Constraints & NULLs.** Short written questions (graded by hand, 4 points each - 12 points total).
5. **Part 5 - Use of AI tools.** One multiple-choice survey question (ungraded).

**How to fill in this notebook:** every spot where you must write an answer is marked
**`YOUR CODE HERE`** — as a `# YOUR CODE HERE` comment in Python code, or a
`-- YOUR CODE HERE` comment inside a SQL string. Replace the marker with your code and
leave everything else in the cell as it is (some cells list requirements or hints as
comments — keep those). For each question:

1. Write your answer in the cell marked `YOUR CODE HERE`.
2. Run that cell.
3. Run the `grader.check(...)` cell just beneath it to self-check your work.

Re-run the **Setup** cell any time you need to rebuild the database from scratch, and
run **every** cell from top to bottom (Kernel → Restart & Run All) before submitting.

## Setup

### Install Otter for the self-checks (one time)

The `grader.check(...)` cells in this notebook use **Otter-Grader** to give you instant
feedback on your answers. Install it once by running the cell below (or run
`pip install otter-grader` in a terminal), then **restart the kernel** so the Setup cell
can find it.

If Otter (or a test) is not available on your machine, the notebook still works — the
check cells simply print a note, and your official grade is computed on Gradescope when
you submit.

In [1]:
# Run this cell ONCE to install Otter for the self-checks, then restart the kernel.
# (Safe to re-run: if Otter is already installed, nothing changes.)
%pip install -q otter-grader

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Setup — run this cell FIRST. Do not edit.
import os, sqlite3, csv
import pandas as pd

# Self-checks. The official grade is produced on Gradescope. Locally, grader.check
# will run only if Otter and the hidden tests are present; otherwise it prints a note.
class _Grader:
    def __init__(self):
        self._nb = None
        try:
            import otter
            self._nb = otter.Notebook()
        except Exception:
            pass
    def check(self, name):
        if self._nb is None:
            print(f"[{name}] Otter is not installed (see the install cell at the top), "
                  "so this self-check was skipped - it will run on Gradescope.")
            return
        try:
            return self._nb.check(name)
        except Exception as e:
            print(f"[{name}] self-check not available locally ({type(e).__name__}); it will run on Gradescope.")
    def check_all(self):
        if self._nb is None: return
        try: return self._nb.check_all()
        except Exception as e: print("check_all not available locally:", e)
    def export(self, *a, **k):
        if self._nb is None: return
        try: return self._nb.export(*a, **k)
        except Exception: pass
grader = _Grader()

DATA_DIR = "data"            # folder containing the provided CSV files
DB_PATH  = "insighta.db"     # the database you will build

# Start from a clean database every run so the notebook is safe to re-run top-to-bottom.
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")   # enforce foreign-key constraints

def run(sql):
    """Run a SQL query against insighta.db and return the result as a DataFrame.
    Returns an empty DataFrame on error so the notebook never aborts mid-run."""
    try:
        return pd.read_sql_query(sql, conn)
    except Exception as e:
        print("Query error:", e)
        return pd.DataFrame()

print("Setup complete. Building database at:", DB_PATH)

Setup complete. Building database at: insighta.db


/Users/berkgozek/Documents/BU-Class-Files/2026 Fall/CDS DS 310/HW/2/hw2 env/lib/python3.14/site-packages/nbformat/validator.py:434: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  _validate(nbdict, ref, version, version_minor, relax_add_props)


## Schema reference

```
User(user_id, email, password, role)
UserPhone(user_id, phone)
Study(study_id, researcher_id, title, study_type, status,
      duration_min, reward_amount, published_at)
ScreeningQuestion(question_id, study_id, question_text, order_num)
Application(application_id, study_id, participant_id, submitted_at, status)
```

| Attribute | Possible values |
|---|---|
| `User.role` | `'researcher'`, `'participant'`, `'both'` |
| `Study.status` | `'active'`, `'paused'`, `'closed'`, `'draft'` |
| `Study.study_type` | `'survey'`, `'interview'` |
| `Application.status` | `'completed'`, `'pending'`, `'approved'`, `'withdrawn'` |

The CSV files in the `data/` folder hold the rows for each table. A blank field in a
CSV means **NULL** (for example, draft studies have an empty `published_at`).

---
## Part 1 - Create the schema

Write `CREATE TABLE` statements that recreate the Insighta schema. Tables are created in the live `insighta.db` database. *(Re-run Setup first if you need to recreate a table.)*

### Q1.1 - Create the User and UserPhone tables

In [3]:
# Q1.1 - Create the User and UserPhone tables
# Requirements (keep these comments; write your SQL below them):
conn.executescript("""
-- User:
--   user_id   : PRIMARY KEY
--   email     : never NULL
--   password, role
-- UserPhone:
--   composite PRIMARY KEY (user_id, phone)
--   FOREIGN KEY (user_id) -> User(user_id)

-- YOUR CODE HERE
    CREATE TABLE User (
        user_id INT PRIMARY KEY,
        email VARCHAR NOT NULL,
        password VARCHAR,
        role VARCHAR
    );
    CREATE TABLE UserPhone (
        user_id INT,
        phone VARCHAR,
        PRIMARY KEY (user_id, phone),
        FOREIGN KEY (user_id) REFERENCES User(user_id)
    );

""")

In [4]:
grader.check("schema_user")

[schema_user] self-check not available locally (ValueError); it will run on Gradescope.


### Q1.2 - Create the Study table

In [5]:
# Q1.2 - Create the Study table
# Requirements (keep these comments; write your SQL below them):
conn.executescript("""
-- study_id      : PRIMARY KEY
-- researcher_id : FOREIGN KEY -> User(user_id)
-- published_at  : may be NULL (drafts)
-- plus title, study_type, status, duration_min, reward_amount

-- YOUR CODE HERE
    CREATE TABLE Study (
        study_id INT PRIMARY KEY,
        researcher_id INT,
        title VARCHAR,
        study_type VARCHAR,
        status VARCHAR,
        duration_min INT,
        reward_amount INT,
        published_at DATETIME,
        
        FOREIGN KEY (researcher_id) REFERENCES User(user_id)
    );

""")

In [6]:
grader.check("schema_study")

[schema_study] self-check not available locally (ValueError); it will run on Gradescope.


### Q1.3 - Create the Application table

In [7]:
# Q1.3 - Create the Application table
# Requirements (keep these comments; write your SQL below them):
conn.executescript("""
-- application_id : PRIMARY KEY
-- UNIQUE (study_id, participant_id)  -- one application per participant per study
-- FOREIGN KEY (study_id)       -> Study(study_id)
-- FOREIGN KEY (participant_id) -> User(user_id)
-- plus submitted_at, status

-- YOUR CODE HERE

    CREATE TABLE Application(
        application_id INT PRIMARY KEY,
        study_id INT,
        participant_id INT,
        
        submitted_at DATETIME,
        status VARCHAR,
        
        CONSTRAINT UQ_study_participant UNIQUE (study_id, participant_id),
        
        FOREIGN KEY (study_id) REFERENCES Study(study_id),
        FOREIGN KEY (participant_id) REFERENCES User(user_id)
    );

""")

In [8]:
grader.check("schema_application")

[schema_application] self-check not available locally (ValueError); it will run on Gradescope.


### Q1.4 - Create the ScreeningQuestion table

In [9]:
# Q1.4 - Create the ScreeningQuestion table
# Requirements (keep these comments; write your SQL below them):
conn.executescript("""
-- question_id : PRIMARY KEY
-- study_id    : FOREIGN KEY -> Study(study_id)
-- plus question_text, order_num

-- YOUR CODE HERE

    CREATE TABLE ScreeningQuestion(
        question_id INT PRIMARY KEY,
        study_id INT,
        
        question_text VARCHAR,
        order_num INT NOT NULL,
        
        FOREIGN KEY (study_id) REFERENCES Study(study_id)
        );
""")

In [10]:
grader.check("schema_screening")

[schema_screening] self-check not available locally (ValueError); it will run on Gradescope.


---
## Part 2 - Load the data with Python + INSERT

The `data/` folder has one CSV per table. The cell below is a **complete, working
loader** — there is nothing for you to write in this part and it is **not graded**.
Read through the code so you understand what it does, then **run it** to fill the
tables you created in Part 1 (so Part 1 must be done first).

Two database calls do the heavy lifting:

- **`conn.executemany(sql, rows)`** runs one SQL statement once **per row** in a list.
  Here `sql` is a *parameterized* `INSERT` — the `?` placeholders (one per column) are
  filled in from each row's values by the database driver itself, which is both safer
  (no quoting bugs / SQL injection) and faster than building the SQL strings yourself.
  A single call inserts an entire table's worth of rows.
- **`conn.commit()`** finalizes the transaction. SQLite groups your `INSERT`s into a
  transaction, and until you commit, the new rows are **not permanently saved** to
  `insighta.db` — forgetting to commit is a classic way to "lose" data that seemed to
  load fine. You will learn more about commit and transactions in later lectures.

Also worth noticing as you read: the loader skips each CSV's header row, converts empty
fields to `NULL`, and loads **parent tables before child tables**, because foreign keys
are enforced.

### Load all five tables *(provided - just run this cell)*

In [11]:
# Part 2 (provided) - Load every CSV into its table. Nothing to write here:
# read through the code, then just RUN this cell. It is not graded.
def load_table(table_name, csv_filename, num_columns):
    path = os.path.join(DATA_DIR, csv_filename)
    with open(path, newline="") as f:
        reader = csv.reader(f)
        next(reader)                                    # skip the header row
        rows = [[None if v == "" else v for v in row]   # empty field -> NULL
                for row in reader]
    placeholders = ", ".join(["?"] * num_columns)       # e.g. "?, ?, ?, ?"
    sql = f"INSERT INTO {table_name} VALUES ({placeholders})"
    conn.executemany(sql, rows)                         # run one INSERT per row

try:
    # Load PARENT tables before CHILD tables (foreign keys are enforced).
    load_table("User",              "user.csv",              4)
    load_table("UserPhone",         "userphone.csv",         2)
    load_table("Study",             "study.csv",             8)
    load_table("ScreeningQuestion", "screeningquestion.csv", 4)
    load_table("Application",       "application.csv",       5)
    conn.commit()                                           # save the transaction
except Exception as e:
    print("Could not load the data:", e)
    print("Most likely Part 1 is not finished yet - create all five tables first,")
    print("then re-run Setup, Part 1, and this cell (in that order).")

def _row_counts():
    out = {}
    for t in ["User", "UserPhone", "Study", "ScreeningQuestion", "Application"]:
        try:
            out[t] = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
        except Exception:
            out[t] = "(table not created yet)"
    return out

print(_row_counts())

{'User': 50, 'UserPhone': 15, 'Study': 20, 'ScreeningQuestion': 30, 'Application': 47}


---
## Part 3 - Query the database

Write each query as a SQL string in the given variable, then run it. The cell displays your result; the `grader.check(...)` cell verifies it.

### Q1 — Filter and sort
List the `title` and `reward_amount` of every **active** study whose reward is **greater than \$10**, sorted by `reward_amount` from highest to lowest.

*Return columns:* `title, reward_amount`  &nbsp;·&nbsp; *(worth 5 points)*

In [12]:
q1 = """
-- YOUR CODE HERE
SELECT title, reward_amount
FROM Study
WHERE reward_amount >10
ORDER BY reward_amount DESC;
"""
run(q1)

,title,reward_amount
0,Exercise & Focus,40
1,Peer Pressure & Decisions,35
2,UX of Insighta Platform,30
3,AI Tools in Education,22
4,Remote Work Productivity,15
5,Internship Quality,15
6,Diversity & Belonging,14
7,Social Media & Mental Health,13
8,Online Learning,13
9,Study Habits & GPA,12


In [13]:
grader.check("q1")

[q1] self-check not available locally (ValueError); it will run on Gradescope.


### Q2 — Pattern matching
List the `email` of every user whose address is a BU address (ends in `@bu.edu`).

*Return columns:* `email`  &nbsp;·&nbsp; *(worth 4 points)*

In [14]:
q2 = """
-- YOUR CODE HERE
SELECT email
FROM User
WHERE email LIKE '%@bu.edu'
"""
run(q2)

,email
0,carol.williams@bu.edu
1,eve.jones@bu.edu
2,frank.garcia@bu.edu
3,grace.miller@bu.edu
4,hank.davis@bu.edu
5,iris.wilson@bu.edu
6,olivia.white@bu.edu
7,eli.taylor30@bu.edu
8,george.thomas32@bu.edu
9,kevin.martin36@bu.edu


In [15]:
grader.check("q2")

[q2] self-check not available locally (ValueError); it will run on Gradescope.


### Q3 — Join with a filter
List the `title` of every study that has received **at least one** `completed` application. Each title should appear **only once**.

*Return columns:* `title (distinct)`  &nbsp;·&nbsp; *(worth 5 points)*

In [16]:
q3 = """
-- YOUR CODE HERE
SELECT title
FROM Study
WHERE study_id IN (
  SELECT study_id
  FROM Application
);
"""
run(q3)

,title
0,Sleep & Academic Performance
1,Social Media & Mental Health
2,Climate Change Attitudes
3,Diversity & Belonging
4,Commute & Wellbeing


In [17]:
grader.check("q3")

[q3] self-check not available locally (ValueError); it will run on Gradescope.


### Q4 — Subquery / set membership
List the `email` of every user who has **never submitted an application**. *Hint: a `NOT IN` subquery works well.*

*Return columns:* `email`  &nbsp;·&nbsp; *(worth 5 points)*

In [18]:
q4 = """
-- YOUR CODE HERE
SELECT email
FROM User
Where User_id NOT IN (
  SELECT participant_id
  FROM Application  
);
"""
run(q4)

,email
0,alice.smith@mit.edu
1,bob.johnson@mit.edu
2,carol.williams@bu.edu
3,dave.brown@harvard.edu
4,eve.jones@bu.edu
5,frank.garcia@bu.edu
6,grace.miller@bu.edu
7,hank.davis@bu.edu
8,iris.wilson@bu.edu
9,jack.moore@mit.edu


In [19]:
grader.check("q4")

[q4] self-check not available locally (ValueError); it will run on Gradescope.


### Q5 — Outer join + aggregate
For **every** study, show its `title` and the **total number of applications** it has received — **including studies with zero applications**. Sort by the count, highest first. *Hint: `LEFT OUTER JOIN` + `GROUP BY`; mind `COUNT(*)` vs `COUNT(column)`.*

*Return columns:* `title, num_apps`  &nbsp;·&nbsp; *(worth 6 points)*

In [20]:
q5 = """
-- YOUR CODE HERE
SELECT Study.title, COUNT(Application.application_id) AS num_apps
FROM Study
LEFT OUTER JOIN Application ON Application.study_id = Study.study_id
GROUP BY Study.study_id, Study.title
ORDER BY num_apps DESC
"""
run(q5)

,title,num_apps
0,Diversity & Belonging,13
1,Commute & Wellbeing,13
2,Sleep & Academic Performance,10
3,Social Media & Mental Health,6
4,Climate Change Attitudes,5
5,UX of Insighta Platform,0
6,Remote Work Productivity,0
7,Food Choices & Stress,0
8,Campus Safety Perceptions,0
9,Exercise & Focus,0


In [21]:
grader.check("q5")

[q5] self-check not available locally (ValueError); it will run on Gradescope.


### Q6 — Group filtering
List the `title` of every study whose applications are **all** `completed` — the study has at least one application and none in any other status.

*Return columns:* `title`  &nbsp;·&nbsp; *(worth 6 points)*

In [22]:
q6 = """
-- YOUR CODE HERE
SELECT Study.title
FROM Study
JOIN Application ON Application.study_id = Study.study_id
GROUP BY Study.study_id, Study.title
HAVING COUNT(*) = SUM(CASE WHEN Application.status = 'completed' THEN 1 ELSE 0 END)
"""
run(q6)

,title
0,Social Media & Mental Health
1,Climate Change Attitudes


In [23]:
grader.check("q6")

[q6] self-check not available locally (ValueError); it will run on Gradescope.


### Q7 — Aggregate in a subquery
Show the `title` and `reward_amount` of the **active** study with the **highest reward**. Use a subquery — do **not** use `ORDER BY ... LIMIT`.

*Return columns:* `title, reward_amount`  &nbsp;·&nbsp; *(worth 5 points)*

In [24]:
q7 = """
-- YOUR CODE HERE
SELECT title, reward_amount
FROM study
WHERE status = 'active'
    AND reward_amount = (
        SELECT MAX(reward_amount)
        FROM Study
        WHERE status = 'active'
    );
"""
run(q7)

,title,reward_amount
0,Exercise & Focus,40


In [25]:
grader.check("q7")

[q7] self-check not available locally (ValueError); it will run on Gradescope.


### Q8 — Counting per group, including zeros
For **every** researcher (every user whose `role` is `'researcher'` or `'both'`), show their `email` and how many studies they have posted — **including researchers who have posted none**.

*Return columns:* `email, num_studies`  &nbsp;·&nbsp; *(worth 5 points)*

In [26]:
q8 = """
-- YOUR CODE HERE
SELECT User.email, COUNT(Study.study_id) AS num_studies
FROM User
LEFT OUTER JOIN Study ON Study.researcher_id = User.user_id
Where User.role IN ('researcher', 'both')
GROUP BY User.user_id, User.email
    

"""
run(q8)

,email,num_studies
0,alice.smith@mit.edu,2
1,bob.johnson@mit.edu,2
2,carol.williams@bu.edu,2
3,dave.brown@harvard.edu,2
4,eve.jones@bu.edu,2
5,frank.garcia@bu.edu,1
6,grace.miller@bu.edu,1
7,hank.davis@bu.edu,1
8,iris.wilson@bu.edu,1
9,jack.moore@mit.edu,1


In [27]:
grader.check("q8")

[q8] self-check not available locally (ValueError); it will run on Gradescope.


### Q9 — HAVING
List the `email` of every participant who has applied to **more than one distinct study**.

*Return columns:* `email`  &nbsp;·&nbsp; *(worth 5 points)*

In [28]:
q9 = """
-- YOUR CODE HERE
SELECT User.email
FROM User
JOIN Application ON Application.participant_id = User.user_id
GROUP BY User.user_id, User.email
HAVING COUNT(DISTINCT Application.study_id) > 1
"""
run(q9)

,email
0,mia.thomas@bc.edu
1,noah.jackson@bc.edu
2,quinn.martin16@northeastern.edu
3,rachel.thompson17@mit.edu
4,tina.walker19@bc.edu
5,uma.smith20@bc.edu
6,yara.jones24@mit.edu
7,chris.wilson28@bc.edu
8,diana.moore29@harvard.edu
9,eli.taylor30@bu.edu


In [29]:
grader.check("q9")

[q9] self-check not available locally (ValueError); it will run on Gradescope.


### Q10 — Grouped average with HAVING
For each `study_type`, compute the **average reward** among **active** studies, and return only the study types whose average exceeds **\$9**.

*Return columns:* `study_type, avg_reward`  &nbsp;·&nbsp; *(worth 5 points)*

In [30]:
q10 = """
-- YOUR CODE HERE
SELECT study_type, AVG(reward_amount) AS avg_reward
FROM Study
WHERE status = 'active'
GROUP BY study_type
HAVING AVG(reward_amount) > 9
"""
run(q10)

,study_type,avg_reward
0,interview,30.666667
1,survey,9.909091


In [31]:
grader.check("q10")

[q10] self-check not available locally (ValueError); it will run on Gradescope.


### Q11 — Filtered outer join
For **every** study, show its `title` and the number of **`completed`** applications it has received — **including studies with zero**. *Think carefully about where the `status = 'completed'` condition must go.*

*Return columns:* `title, completed_count`  &nbsp;·&nbsp; *(worth 6 points)*

In [32]:
q11 = """
-- YOUR CODE HERE
SELECT Study.title, COUNT(Application.application_id) AS completed_count
FROM Study
LEFT OUTER JOIN Application
    ON Application.study_id = Study.study_id
    AND Application.status = 'completed'
GROUP BY Study.study_id, Study.title
"""
run(q11)

,title,completed_count
0,Sleep & Academic Performance,5
1,Social Media & Mental Health,6
2,UX of Insighta Platform,0
3,Climate Change Attitudes,5
4,Remote Work Productivity,0
5,Food Choices & Stress,0
6,Campus Safety Perceptions,0
7,Exercise & Focus,0
8,AI Tools in Education,0
9,Financial Stress in College,0


In [33]:
grader.check("q11")

[q11] self-check not available locally (ValueError); it will run on Gradescope.


---
## Part 4 - Constraints & NULLs (written, manually graded)

Answer in the Markdown cells below. These questions are graded by hand - there is no
autograder check for them. **Each question (Q4.1-Q4.3) is worth 4 points (12 points total).** Foreign keys **are enabled** in this notebook
(`PRAGMA foreign_keys = ON`), so reason about the constraints as actually enforced.

### Q4.1 - Allow or reject?  *(worth 4 points)*
For each operation, state **ALLOW** or **REJECT** and give a one-sentence reason.

**A.** `INSERT INTO Application VALUES (9999, 202, 16, '2025-06-01', 'pending')` - *participant 16 has not yet applied to study 202.* 

**B.** `INSERT INTO Application VALUES (9998, 202, 16, '2025-06-01', 'pending')` - *now participant 16 already has an application for study 202.*

**C.** `INSERT INTO Application VALUES (9997, 999, 16, '2025-06-01', 'pending')`

**D.** `DELETE FROM Study WHERE study_id = 201`

*Your answers:*

- A. ALLOW, all FKs exist and (202, 16) is not yet in the unique pair.
- B. REJECT, UNIQUE (study_id, participant_id) is already taken after A.
- C. REJECT, no Study row with study_id = 999.
- D. REJECT, foreign keys are on and 201 still has applications.

### Q4.2 - NULLs  *(worth 4 points)*
Studies 217 and 218 are drafts with `published_at = NULL`.

**A.** Will the query below return studies 217 or 218? Explain why or why not.
```sql
SELECT title FROM Study WHERE published_at != '2025-01-10';
```
**B.** Write (in the code cell) a query returning all studies that have **not yet been published**.

*Your answer to A:*

No. 217 and 218 will not show up.

WHERE only keeps rows where the condition is true. For those drafts, published_at is NULL, and NULL != '2025-01-10' is unknown, not true. SQL does not treat “not equal” as covering missing values.



In [34]:
# Q4.2B - write and run your query
q4_2b = """
-- YOUR CODE HERE
SELECT *
FROM Study
WHERE published_at IS NULL
"""
run(q4_2b)

,study_id,researcher_id,title,study_type,status,duration_min,reward_amount,published_at
0,217,3,Housing Insecurity,survey,draft,18,10,None
1,218,4,Internship Quality,survey,draft,25,15,None


### Q4.3 - WHERE vs ON  *(worth 4 points)*
A classmate counts non-withdrawn applications per study like this:
```sql
SELECT study_id, COUNT(*)
FROM Application
WHERE status != 'withdrawn'
GROUP BY study_id;
```
This **omits studies that have zero non-withdrawn applications**.

**A.** Explain (i) why a `LEFT OUTER JOIN` from `Study` is needed at all, and (ii) why the `status != 'withdrawn'` condition must go in the `ON` clause rather than the `WHERE` clause.

**B.** Write the corrected query (in the code cell): every study with its count of non-withdrawn applications, including zeros.

*Your answer to A:*

i - Start from Study with a LEFT OUTER JOIN. Application only has studies that received at least one application, so grouping that table can never show a study with a count of 0.

ii - ON decides which application rows attach. WHERE decides which result rows survive. After a left join, a study with no non-withdrawn apps has status as NULL. NULL != 'withdrawn' is not true, so WHERE deletes that study. Put the filter in ON and the study still appears, with no application attached.

In [35]:
# Q4.3B - write and run your corrected query
q4_3b = """
-- YOUR CODE HERE
SELECT Study.study_id, COUNT(Application.application_id) AS num_apps
FROM Study
LEFT OUTER JOIN Application
    ON Application.study_id = Study.study_id
    AND Application.status != 'withdrawn'
GROUP BY Study.study_id
"""
run(q4_3b)

,study_id,num_apps
0,200,8
1,201,6
2,202,0
3,203,5
4,204,0
5,205,0
6,206,0
7,207,0
8,208,0
9,209,0


---
## Part 5 - Use of AI tools (ungraded)

This question is **not graded** and has no effect on your score - please answer honestly.
It helps us understand how AI tools are being used in the course.

### Q5.1 - Which of the following describe how you used an AI assistant (ChatGPT, Claude, Copilot, ...) on this assignment?

**Select all that apply** - you may choose more than one option. Write the letter(s) below.

- **(a)** Not at all
- **(b)** To check my work
- **(c)** To explain my mistakes
- **(d)** To generate answers

*Your answer (one or more of a, b, c, d):*
b

---
## Submit
Run **all** cells from top to bottom (Kernel → Restart & Run All), make sure every check passes, then upload **this notebook** (`.ipynb`) to Gradescope.

In [36]:
grader.check_all()